# DA-GPS vs OpenDSS daily compare — Google Colab (CUDA GPU)

Runs `nonunique.ipynb` cell 2 (`mode="da_gps_daily_compare"`): the DA-GPS GNN vs
OpenDSS native daily QSTS truth, on a Colab CUDA GPU.

**Before you start:** open `Runtime > Change runtime type > GPU` (T4 is fine).

Run the cells top to bottom. The GNN auto-detects CUDA and uses the GPU.

## What ships in git — no upload needed
- **In git (cloned automatically):** all code modules, the grid DSS folder, the
  reference load/PV profiles, the edge CSV, the small model checkpoint
  (`training_last.pt` + norm/sidecar tensors, ~9 MB), **and** the slim
  single-sample DA-GPS tensor cache
  `run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt` (~0.28 MB).
- The slim cache is bit-identical to the old 359 MB full cache for daily-compare
  inference (which only ever reads sample `ref_sample_index=0`), so **there is
  nothing to upload to Drive anymore.** Section 4 below is optional/legacy.

## 1. Clone the repo
If the GitHub repo is **private**, replace the URL with a token form, e.g.
`https://<USERNAME>:<PERSONAL_ACCESS_TOKEN>@github.com/alitasavori/GNN-Sandia.git`.

In [ ]:
REPO_URL = "https://github.com/alitasavori/GNN-Sandia.git"  # private? use a token URL
REPO_DIR = "/content/GNN2"

import os
if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("Repo already cloned at", REPO_DIR)
%cd $REPO_DIR
!git log --oneline -3

## 2. Install dependencies
Colab already provides a CUDA-enabled `torch`. `torch_geometric` (>=2.4) needs no
external `torch-scatter`/`torch-sparse` for this model (only `GINEConv`/`Data`).

In [ ]:
!pip install -q torch_geometric "opendssdirect.py"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Portable paths + GPU env
`GNN2_REPO_ROOT` makes the modules resolve all data/checkpoint paths to the cloned
repo. `GNN_TORCH_COMPILE=0` skips `torch.compile` (faster startup on Colab).

In [ ]:
import os
os.environ["GNN2_REPO_ROOT"] = "/content/GNN2"
os.environ["GNN_TORCH_COMPILE"] = "0"

## 4. (Optional / legacy) Fetch the full 359 MB tensor cache
**You do not need this for the daily compare.** The slim cache
`run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt` ships in git and is the
default the loader uses (`DA_GPS_CACHE_PT` in `nonunique_opendss_daily.py`). The
cell below just verifies the slim cache is present after the clone.

Only set `CACHE_DRIVE_FILE_ID` if you specifically need the *full* multi-sample
cache (e.g. for training/eval over all samples). It will be downloaded to
`datasets_gnn2_from pc/run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt`.

In [ ]:
import os

# Default (used by the daily compare): slim single-sample cache shipped in git.
SLIM_REL = (
    "datasets_gnn2_from pc/"
    "run_001_ref0_slim__full__nobess__regce__mauxb7bd1d58.pt"
)
SLIM_PATH = os.path.join("/content/GNN2", SLIM_REL)
if os.path.isfile(SLIM_PATH):
    print("Slim cache present (no upload needed):", SLIM_PATH,
          round(os.path.getsize(SLIM_PATH) / 1e6, 3), "MB")
else:
    print("WARNING: slim cache missing at", SLIM_PATH,
          "- re-clone the repo or run make_slim_da_gps_cache.py")

# --- Optional / legacy: only needed if you want the FULL multi-sample cache ---
CACHE_DRIVE_FILE_ID = ""  # leave "" to skip; paste a Drive file ID to fetch the full cache
CACHE_REL = (
    "datasets_gnn2_from pc/"
    "run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt"
)
CACHE_PATH = os.path.join("/content/GNN2", CACHE_REL)
if CACHE_DRIVE_FILE_ID:
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    if os.path.isfile(CACHE_PATH) and os.path.getsize(CACHE_PATH) > 100_000_000:
        print("Full cache already present:", CACHE_PATH)
    else:
        !pip install -q gdown
        import gdown
        gdown.download(id=CACHE_DRIVE_FILE_ID, output=CACHE_PATH, quiet=False)
        print("Downloaded ->", CACHE_PATH, os.path.getsize(CACHE_PATH) / 1e6, "MB")

### (Alternative) Mount Google Drive instead of gdown
If you prefer mounting Drive, run this instead of the cell above and adjust the
source path to wherever you stored the file.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os
# SRC = '/content/drive/MyDrive/run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt'
# os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
# shutil.copy(SRC, CACHE_PATH)
# print('copied ->', CACHE_PATH)

## 5. Run the compare (quick smoke first)
`npts=12` is a fast sanity check. Once it works, run the full-day cell below.

In [ ]:
%matplotlib inline
import sys
for _m in (
    "nonunique_daily_experiment", "nonunique_da_gps_daily_compare",
    "nonunique_warmstart_compare", "nonunique_four_scenario_demo",
    "nonunique_opendss_daily", "nonunique_da_gps", "nonunique_plots",
    "run_da_gps_daily_opendss_compare",
):
    sys.modules.pop(_m, None)
from nonunique_daily_experiment import run_and_plot

# Quick smoke: 12 points. DA-GPS auto-uses CUDA when available.
run_and_plot(
    mode="da_gps_daily_compare",
    step_min=5,
    npts=12,
    include_der=False,
    include_da_gps=True,
    show=True,
)

## 6. Full-day run (288 points @ 5 min)
Drop the `npts` override to run the full day, matching `nonunique.ipynb` cell 2.

In [ ]:
%matplotlib inline
import sys
for _m in (
    "nonunique_daily_experiment", "nonunique_da_gps_daily_compare",
    "nonunique_warmstart_compare", "nonunique_four_scenario_demo",
    "nonunique_opendss_daily", "nonunique_da_gps", "nonunique_plots",
    "run_da_gps_daily_opendss_compare",
):
    sys.modules.pop(_m, None)
from nonunique_daily_experiment import run_and_plot

run_and_plot(
    mode="da_gps_daily_compare",
    step_min=5,
    include_der=False,
    include_da_gps=True,
    show=True,
)